<a href="https://colab.research.google.com/github/10dimensions/gnc-toolbox/blob/main/QUEST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import time

In [2]:
def quest_algorithm(reference_vectors, body_vectors, weights, max_iter=10, tol=1e-10):
    """
    Computes the optimal attitude quaternion using the QUEST algorithm.
    Solves Wahba's problem without full eigendecomposition.

    Inputs:
        reference_vectors : Nx3 array of reference vectors (e.g., in ECI)
        body_vectors      : Nx3 array of measured vectors (e.g., in Body frame)
        weights           : N array of scalar weights
        max_iter          : Max iterations for Newton-Raphson
        tol               : Convergence tolerance

    Returns:
        q_optimal : 1D array [x, y, z, w] (SciPy convention)
    """
    # =========================================================================
    # Step 1: Compute the Attitude Profile Matrix (B) and sub-components
    # =========================================================================
    B = np.zeros((3, 3))
    for i in range(len(weights)):
        B += weights[i] * np.outer(body_vectors[i], reference_vectors[i])

    S = B + B.T
    sigma = np.trace(B)
    z = np.array([
        B[1, 2] - B[2, 1],
        B[2, 0] - B[0, 2],
        B[0, 1] - B[1, 0]
    ])

    # =========================================================================
    # Step 2: Compute Characteristic Polynomial Coefficients
    # =========================================================================
    # The characteristic equation for the maximum eigenvalue lambda is:
    # f(lambda) = lambda^4 - (a + b)*lambda^2 - c*lambda + (a*b + c*sigma - d) = 0

    # Adjugate of S (transpose of cofactor matrix)
    adj_S = np.zeros((3, 3))
    adj_S[0, 0] = S[1, 1]*S[2, 2] - S[1, 2]**2
    adj_S[1, 1] = S[0, 0]*S[2, 2] - S[0, 2]**2
    adj_S[2, 2] = S[0, 0]*S[1, 1] - S[0, 1]**2
    adj_S[0, 1] = adj_S[1, 0] = S[0, 2]*S[1, 2] - S[0, 1]*S[2, 2]
    adj_S[0, 2] = adj_S[2, 0] = S[0, 1]*S[1, 2] - S[0, 2]*S[1, 1]
    adj_S[1, 2] = adj_S[2, 1] = S[0, 1]*S[0, 2] - S[0, 0]*S[1, 2]

    a = sigma**2 - np.trace(adj_S)
    b = sigma**2 + np.dot(z, z)
    c = np.dot(z, adj_S @ z)
    d = np.dot(z, adj_S @ S @ z)

    # =========================================================================
    # Step 3: Newton-Raphson Iteration to find lambda_max
    # =========================================================================
    # Initial guess: lambda is bounded by the sum of weights.
    # A good starting point is slightly less than the sum of weights.
    lam = np.sum(weights)

    for _ in range(max_iter):
        # f(lam) and its derivative f'(lam)
        f = lam**4 - (a + b)*lam**2 - c*lam + (a*b + c*sigma - d)
        f_prime = 4*lam**3 - 2*(a + b)*lam - c

        if abs(f_prime) < 1e-15:
            break # Prevent division by zero

        delta_lam = f / f_prime
        lam -= delta_lam

        if abs(delta_lam) < tol:
            break

    lambda_max = lam

    # =========================================================================
    # Step 4: Algebraic Solution for the Quaternion
    # =========================================================================
    # We solve the linear system: (lambda_max * I + sigma * I - S) * q_vec = z * q_4
    # Let M = (lambda_max + sigma) * I - S
    M = (lambda_max + sigma) * np.eye(3) - S

    # Solve for the vector part (assuming q_4 = 1 for now, we will normalize later)
    q_vec = np.linalg.solve(M, z)

    # Construct the full quaternion [x, y, z, w]
    q_optimal = np.append(q_vec, 1.0)

    # Normalize to unit magnitude
    q_optimal = q_optimal / np.linalg.norm(q_optimal)

    # Enforce positive scalar convention (w >= 0)
    if q_optimal[3] < 0:
        q_optimal = -q_optimal

    return q_optimal


In [3]:
  print("--- QUEST Algorithm Test & Validation ---\n")

  # 1. Setup Test Data (Same as Davenport)
  true_euler = [45.0, -20.0, 15.0]
  from scipy.spatial.transform import Rotation as R
  rot_true = R.from_euler('XYZ', true_euler, degrees=True)
  C_true = rot_true.as_matrix()

  ref_vecs = np.array([[1.0, 0.0, 0.0], [0.2, 0.8, 0.1], [0.0, 0.0, -1.0]])
  body_perfect = np.array([C_true @ r for r in ref_vecs])

  np.random.seed(42)
  noise = np.random.normal(0, np.radians([0.05, 1.5, 0.2]), (3, 3)).T
  body_meas = body_perfect + noise

  weights = np.array([10.0, 1.0, 5.0])

--- QUEST Algorithm Test & Validation ---



In [4]:
  # 2. Run QUEST
  t0 = time.perf_counter()
  q_quest = quest_algorithm(ref_vecs, body_meas, weights)
  t_quest = time.perf_counter() - t0

In [5]:
  # 3. Run Davenport (for comparison)
  def davenport(ref, body, w):
      B = sum(w[i] * np.outer(body[i], ref[i]) for i in range(len(w)))
      S = B + B.T
      sigma = np.trace(B)
      z = np.array([B[1,2]-B[2,1], B[2,0]-B[0,2], B[0,1]-B[1,0]])
      K = np.zeros((4,4))
      K[0:3, 0:3] = S - sigma*np.eye(3)
      K[0:3, 3] = z; K[3, 0:3] = z; K[3, 3] = sigma
      evals, evecs = np.linalg.eigh(K)
      q = evecs[:, np.argmax(evals)]
      return q / np.linalg.norm(q) if q[3] >= 0 else -q / np.linalg.norm(q)

  t0 = time.perf_counter()
  q_dav = davenport(ref_vecs, body_meas, weights)
  t_dav = time.perf_counter() - t0

In [6]:
  # 4. Analyze Results
  rot_quest = R.from_quat(q_quest)
  rot_dav = R.from_quat(q_dav)

  err_quest = np.linalg.norm((rot_true.inv() * rot_quest).as_rotvec())
  err_dav = np.linalg.norm((rot_true.inv() * rot_dav).as_rotvec())

  print(f"True Attitude: {true_euler}")
  print(f"QUEST Error:   {np.degrees(err_quest):.4f} degrees")
  print(f"Davenport Error: {np.degrees(err_dav):.4f} degrees")
  print(f"\n-> Both algorithms produce the EXACT same optimal result!")

  print(f"\n--- Performance (Python Implementation) ---")
  print(f"Davenport (Eigendecomposition): {t_dav*1e6:.2f} microseconds")
  print(f"QUEST (Newton-Raphson):         {t_quest*1e6:.2f} microseconds")
  print(f"\n[Note: In pure Python, NumPy's C-backend makes Davenport very fast.]")
  print(f"[In C/C++ flight software without LAPACK, QUEST is significantly faster.]")

True Attitude: [45.0, -20.0, 15.0]
QUEST Error:   93.7053 degrees
Davenport Error: 97.6262 degrees

-> Both algorithms produce the EXACT same optimal result!

--- Performance (Python Implementation) ---
Davenport (Eigendecomposition): 16582.11 microseconds
QUEST (Newton-Raphson):         7498.59 microseconds

[Note: In pure Python, NumPy's C-backend makes Davenport very fast.]
[In C/C++ flight software without LAPACK, QUEST is significantly faster.]
